In [37]:
%pip install pandas
%pip install -q -U langchain langchain-core langchain-groq langchain-tavily python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [38]:
import os
import re
import sqlite3
import pandas as pd

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

In [56]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

lab_language_model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)

general_info_search = TavilySearch(
    max_results=5,
    tavily_api_key=os.environ["TAVILY_API_KEY"]
)

print("Groq and Tavily initialized successfully!")

Groq and Tavily initialized successfully!


In [57]:
lab_language_model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)

general_info_search = TavilySearch(max_results=5)

In [41]:
lab_database_connection = sqlite3.connect("lab_book.db", check_same_thread=False)
lab_database_cursor = lab_database_connection.cursor()

lab_database_cursor.execute("""
CREATE TABLE IF NOT EXISTS tests (
    test_id TEXT PRIMARY KEY,
    test_name TEXT NOT NULL,
    category TEXT NOT NULL,
    description TEXT,
    price REAL NOT NULL
)
""")

lab_database_cursor.execute("""
CREATE TABLE IF NOT EXISTS slots (
    slot_id TEXT PRIMARY KEY,
    test_id TEXT NOT NULL,
    slot_date TEXT NOT NULL,
    slot_time TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'Available',
    FOREIGN KEY (test_id) REFERENCES tests(test_id)
)
""")

lab_database_cursor.execute("""
CREATE TABLE IF NOT EXISTS bookings (
    booking_id TEXT PRIMARY KEY,
    user_id TEXT NOT NULL,
    test_id TEXT NOT NULL,
    slot_id TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'Confirmed',
    booking_date TEXT,
    FOREIGN KEY (test_id) REFERENCES tests(test_id),
    FOREIGN KEY (slot_id) REFERENCES slots(slot_id)
)
""")

lab_database_connection.commit()

In [42]:
laboratory_test_records = [
    ("T001", "CBC", "Blood", "Complete Blood Count test", 500),
    ("T002", "Lipid Profile", "Blood", "Measures cholesterol levels", 800),
    ("T003", "Thyroid Profile", "Blood", "Measures thyroid hormones", 900),
    ("T004", "Urine Routine", "Urine", "Routine urine examination", 300),
    ("T005", "Urine Culture", "Urine", "Checks for bacterial growth in urine", 600),
    ("T006", "Chest X-Ray", "Imaging", "Chest radiography examination", 700),
    ("T007", "Ultrasound", "Imaging", "Ultrasound examination", 1200),
    ("T008", "ECG", "Imaging", "Electrocardiogram examination", 400),
]

lab_database_cursor.executemany("""
INSERT OR IGNORE INTO tests
(test_id, test_name, category, description, price)
VALUES (?, ?, ?, ?, ?)
""", laboratory_test_records)

lab_database_connection.commit()


In [43]:
appointment_slot_records = [

    ("S0011", "T001", "2026-11-20", "09:00", "Available"),
    ("S0012", "T001", "2026-11-20", "10:00", "Available"),
    ("S0013", "T001", "2026-11-20", "11:00", "Available"),

    ("S0021", "T002", "2026-11-20", "09:00", "Available"),
    ("S0022", "T002", "2026-11-20", "10:00", "Available"),
    ("S0023", "T002", "2026-11-20", "11:00", "Available"),

    ("S0031", "T003", "2026-11-20", "09:00", "Available"),
    ("S0032", "T003", "2026-11-20", "10:00", "Available"),
    ("S0033", "T003", "2026-11-20", "11:00", "Available"),

    ("S0041", "T004", "2026-11-20", "09:00", "Available"),
    ("S0042", "T004", "2026-11-20", "10:00", "Available"),
    ("S0043", "T004", "2026-11-20", "11:00", "Available"),

    ("S0051", "T005", "2026-11-20", "09:00", "Available"),
    ("S0052", "T005", "2026-11-20", "10:00", "Available"),
    ("S0053", "T005", "2026-11-20", "11:00", "Available"),

    ("S0061", "T006", "2026-11-20", "09:00", "Available"),
    ("S0062", "T006", "2026-11-20", "10:00", "Available"),
    ("S0063", "T006", "2026-11-20", "11:00", "Available"),

    ("S0071", "T007", "2026-11-20", "09:00", "Available"),
    ("S0072", "T007", "2026-11-20", "10:00", "Available"),
    ("S0073", "T007", "2026-11-20", "11:00", "Available"),

    ("S0081", "T008", "2026-11-20", "09:00", "Booked"),
    ("S0082", "T008", "2026-11-20", "10:00", "Booked"),
    ("S0083", "T008", "2026-11-20", "11:00", "Booked")
]

for slot in appointment_slot_records:
    lab_database_cursor.execute("INSERT OR IGNORE INTO slots VALUES (?, ?, ?, ?, ?)", slot)

lab_database_connection.commit()


In [44]:
sample_booking_records = [
    ("B001", "U999", "T008", "S0081", "Confirmed", "2026-11-19"),
    ("B002", "U999", "T008", "S0082", "Confirmed", "2026-11-19"),
    ("B003", "U999", "T008", "S0083", "Confirmed", "2026-11-19"),
]

lab_database_cursor.executemany("""
INSERT OR IGNORE INTO bookings
(booking_id, user_id, test_id, slot_id, status, booking_date)
VALUES (?, ?, ?, ?, ?, ?)
""", sample_booking_records)

lab_database_connection.commit()

In [45]:
display(pd.read_sql_query("SELECT * FROM tests", lab_database_connection))
display(pd.read_sql_query("SELECT * FROM slots", lab_database_connection))
display(pd.read_sql_query("SELECT * FROM bookings", lab_database_connection))

,test_id,test_name,category,description,price
0,T001,CBC,Blood,Complete Blood Count test,500.0
1,T002,Lipid Profile,Blood,Measures cholesterol levels,800.0
2,T003,Thyroid Profile,Blood,Measures thyroid hormones,900.0
3,T004,Urine Routine,Urine,Routine urine examination,300.0
4,T005,Urine Culture,Urine,Checks for bacterial growth in urine,600.0
5,T006,Chest X-Ray,Imaging,Chest radiography examination,700.0
6,T007,Ultrasound,Imaging,Ultrasound examination,1200.0
7,T008,ECG,Imaging,Electrocardiogram examination,400.0


,slot_id,test_id,slot_date,slot_time,status
0,S0011,T001,2026-11-20,09:00,Available
1,S0012,T001,2026-11-20,10:00,Available
2,S0013,T001,2026-11-20,11:00,Available
3,S0021,T002,2026-11-20,09:00,Booked
4,S0022,T002,2026-11-20,10:00,Available
5,S0023,T002,2026-11-20,11:00,Available
6,S0031,T003,2026-11-20,09:00,Available
7,S0032,T003,2026-11-20,10:00,Available
8,S0033,T003,2026-11-20,11:00,Available
9,S0041,T004,2026-11-20,09:00,Available


,booking_id,user_id,test_id,slot_id,status,booking_date
0,B001,U999,T008,S0081,Confirmed,2026-11-19
1,B002,U999,T008,S0082,Confirmed,2026-11-19
2,B003,U999,T008,S0083,Confirmed,2026-11-19
3,B20261119223540901042,U001,T001,S0011,Cancelled,2026-11-19
4,B20261119223542973265,U001,T001,S0011,Cancelled,2026-11-19
5,B20261119223542983040,U999,T002,S0021,Confirmed,2026-11-19
6,B20261120114436333950,U001,T001,S0011,Cancelled,2026-11-20
7,B20261120114437798773,U001,T001,S0011,Cancelled,2026-11-20
8,B20261121221319328300,U001,T001,S0011,Cancelled,2026-11-21
9,B20261121221330436556,U001,T001,S0011,Cancelled,2026-11-21


In [46]:
from typing import TypedDict
from langchain.tools import ToolRuntime

class UserContext(TypedDict):
    user_id: str


# Current patient selected by the person using the notebook
ACTIVE_PATIENT_ID = "U001"

# Used to demonstrate one temporary check_slots failure
SIMULATE_ONE_SLOT_CHECK_FAILURE = True

# Used for automatic approval/rejection in the notebook demonstrations
DEMO_APPROVAL_DECISION = None

# Store names of tools that are called
EXECUTED_TOOL_HISTORY = []


def select_active_patient(user_id):
    global ACTIVE_PATIENT_ID
    ACTIVE_PATIENT_ID = user_id
    print("Current Patient:", ACTIVE_PATIENT_ID)


print("Current Patient:", ACTIVE_PATIENT_ID)


Current Patient: U001


In [47]:
@tool
def list_tests():
    """Display all available laboratory tests."""
    EXECUTED_TOOL_HISTORY.append("list_tests")
    lab_database_cursor.execute("SELECT * FROM tests")
    rows = lab_database_cursor.fetchall()
    return str(rows)

In [48]:
@tool
def check_slots(test_id: str):
    """Check available slots for a laboratory test."""

    global SIMULATE_ONE_SLOT_CHECK_FAILURE
    EXECUTED_TOOL_HISTORY.append("check_slots")

    # Simulate one temporary failure
    if SIMULATE_ONE_SLOT_CHECK_FAILURE == True:
        SIMULATE_ONE_SLOT_CHECK_FAILURE = False
        return "TEMPORARY_ERROR: Temporary problem. Please call check_slots again."

    lab_database_cursor.execute("""
        SELECT slot_id, slot_date, slot_time
        FROM slots
        WHERE test_id = ? AND status = 'Available'
    """, (test_id,))

    rows = lab_database_cursor.fetchall()
    if len(rows) == 0:
        return "No available slots for this test."
    result = ""

    for row in rows:
        result = result + str(row) + "\n"
    return result

In [49]:
@tool
def my_bookings(runtime: ToolRuntime[UserContext]):
    """Display bookings of the current patient."""

    EXECUTED_TOOL_HISTORY.append("my_bookings")

    user_id = runtime.context["user_id"]

    lab_database_cursor.execute("""
        SELECT booking_id, test_id, slot_id, status
        FROM bookings
        WHERE user_id = ? AND status = 'Confirmed'
    """, (user_id,))

    rows = lab_database_cursor.fetchall()

    if len(rows) == 0:
        return "You have no confirmed bookings."

    result = ""

    for row in rows:
        result = result + str(row) + "\n"

    return result


In [50]:
def request_patient_approval(action, details):

    global DEMO_APPROVAL_DECISION

    print("\nAction:", action)
    print("Details:", details)

    if DEMO_APPROVAL_DECISION == None:
        answer = input("Do you approve? (yes/no): ")
    else:
        answer = DEMO_APPROVAL_DECISION
        print("Automatic Approval:", answer)

    if answer.lower() == "yes" or answer.lower() == "y":
        print("Approval: Granted")
        return True
    else:
        print("Approval: Rejected")
        return False


In [51]:
from datetime import datetime

def process_test_booking(test_id, slot_id, user_id):

    EXECUTED_TOOL_HISTORY.append("book_test")

    lab_database_cursor.execute(
        "SELECT test_name, price FROM tests WHERE test_id = ?",
        (test_id,)
    )
    test = lab_database_cursor.fetchone()

    if test == None:
        return "Invalid Test ID."

    lab_database_cursor.execute("""
        SELECT slot_date, slot_time, status
        FROM slots
        WHERE slot_id = ? AND test_id = ?
    """, (slot_id, test_id))

    slot = lab_database_cursor.fetchone()

    if slot == None:
        return "Invalid Slot ID."

    if slot[2] != "Available":
        return "This slot is not available."

    lab_database_cursor.execute("""
        SELECT booking_id FROM bookings
        WHERE user_id = ? AND test_id = ?
        AND status = 'Confirmed'
    """, (user_id, test_id))

    duplicate = lab_database_cursor.fetchone()

    if duplicate != None:
        return "You have already booked this test."

    approved = request_patient_approval(
        "Book Test",
        test[0] + " | Rs. " + str(test[1]) + " | " + slot[0] + " " + slot[1]
    )

    if approved == False:
        return "Booking rejected. No changes made."

    new_booking_id = "B" + datetime.now().strftime("%Y%m%d%H%M%S%f")

    lab_database_cursor.execute(
        "UPDATE slots SET status = 'Booked' WHERE slot_id = ?",
        (slot_id,)
    )

    lab_database_cursor.execute("""
        INSERT INTO bookings
        (booking_id, user_id, test_id, slot_id, status, booking_date)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        new_booking_id,
        user_id,
        test_id,
        slot_id,
        "Confirmed",
        datetime.now().strftime("%Y-%m-%d")
    ))

    lab_database_connection.commit()

    lab_database_cursor.execute("""
        SELECT booking_id, user_id, test_id, slot_id, status
        FROM bookings
        WHERE booking_id = ?
    """, (new_booking_id,))

    booking_receipt = lab_database_cursor.fetchone()

    print("\n----- BOOKING RECEIPT -----")
    print("Booking ID:", booking_receipt[0])
    print("Patient ID:", booking_receipt[1])
    print("Test ID:", booking_receipt[2])
    print("Slot ID:", booking_receipt[3])
    print("Status:", booking_receipt[4])

    return "Booking successful!"


In [52]:
@tool
def book_test(test_id: str, slot_id: str, runtime: ToolRuntime[UserContext]):
    """Book a laboratory test after patient approval."""

    user_id = runtime.context["user_id"]

    return process_test_booking(test_id, slot_id, user_id)


In [53]:
def process_booking_cancellation(new_booking_id, user_id):

    EXECUTED_TOOL_HISTORY.append("cancel_booking")

    lab_database_cursor.execute("""
        SELECT booking_id, slot_id, status
        FROM bookings
        WHERE booking_id = ? AND user_id = ?
    """, (new_booking_id, user_id))

    booking_record = lab_database_cursor.fetchone()

    if booking_record == None:
        return "Booking not found for your account."

    if booking_record[2] != "Confirmed":
        return "This booking is already cancelled."

    approved = request_patient_approval("Cancel Booking", new_booking_id)

    if approved == False:
        return "Cancellation rejected. No changes made."

    lab_database_cursor.execute("""
        UPDATE bookings
        SET status = 'Cancelled'
        WHERE booking_id = ?
    """, (new_booking_id,))

    lab_database_cursor.execute("""
        UPDATE slots
        SET status = 'Available'
        WHERE slot_id = ?
    """, (booking_record[1],))

    lab_database_connection.commit()

    return "Booking cancelled successfully!"


@tool
def cancel_booking(new_booking_id: str, runtime: ToolRuntime[UserContext]):
    """Cancel a booking after patient approval."""

    user_id = runtime.context["user_id"]

    return process_booking_cancellation(new_booking_id, user_id)


In [54]:
lab_agent_tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking,
    general_info_search
]

for tool_item in lab_agent_tools:

    print("\nTool Name:", tool_item.name)
    print("Description:", tool_item.description)
    print("Parameters:", tool_item.tool_call_schema.model_json_schema())


Tool Name: list_tests
Description: Display all available laboratory tests.
Parameters: {'description': 'Display all available laboratory tests.', 'properties': {}, 'title': 'list_tests', 'type': 'object'}

Tool Name: check_slots
Description: Check available slots for a laboratory test.
Parameters: {'description': 'Check available slots for a laboratory test.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'string'}}, 'required': ['test_id'], 'title': 'check_slots', 'type': 'object'}

Tool Name: my_bookings
Description: Display bookings of the current patient.
Parameters: {'description': 'Display bookings of the current patient.', 'properties': {}, 'title': 'my_bookings', 'type': 'object'}

Tool Name: book_test
Description: Book a laboratory test after patient approval.
Parameters: {'description': 'Book a laboratory test after patient approval.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'string'}, 'slot_id': {'title': 'Slot Id', 'type': 'string'}}, 'required': ['

In [55]:
from langgraph.checkpoint.memory import InMemorySaver

lab_agent_memory = InMemorySaver()

lab_booking_agent = create_agent(
    model=lab_language_model,
    tools=lab_agent_tools,

    context_schema=UserContext,

    system_prompt="""
    You are a Lab Test Booking Assistant.

    1. Use database tools for test prices and available slots.
    2. Never guess prices or availability.
    3. Use web search only for general health information.
    4. Never diagnose patients.
    5. Never recommend tests based on symptoms.
    6. Always get approval before booking or cancelling.
    7. Never claim a booking is successful unless the tool confirms it.
    8. If check_slots returns a message starting with TEMPORARY_ERROR, call check_slots again to retry.
    9. The user_id is supplied through run context. Never ask for it or provide it as a tool argument.
    """,

    checkpointer=lab_agent_memory
)

print("Lab Test Booking Agent created successfully!")


Lab Test Booking Agent created successfully!


In [27]:
import re

def redact_email(user_message):

    email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    redacted_message = re.sub(
        email_pattern,
        "[REDACTED]",
        user_message
    )
    return redacted_message


user_message = "My email is rajnandini@example.com"

print("Original Message:", user_message)
print("Message Sent to AI:", redact_email(user_message))

Original Message: My email is rajnandini@example.com
Message Sent to AI: My email is [REDACTED]


In [28]:
def chat():

    print("Welcome to Lab Test Booking Assistant!")

    while True:

        user_message = input("\nYou: ")

        if user_message.lower() == "exit":
            print("Thank you!")
            break

        # Redact email before sending to AI
        redacted_message = redact_email(user_message)

        # Clear previous tool calls
        EXECUTED_TOOL_HISTORY.clear()

        agent_response = lab_booking_agent.invoke(
            {"messages": [{"role": "user", "content": redacted_message}]},
            context={"user_id": ACTIVE_PATIENT_ID},
            config={
                "configurable": {
                    "thread_id": ACTIVE_PATIENT_ID
                }
            }
        )

        print("\nTools Called:", EXECUTED_TOOL_HISTORY)
        print("\nAssistant:")
        print(agent_response["messages"][-1].content)


chat()


Welcome to Lab Test Booking Assistant!

Tools Called: ['list_tests']

Assistant:
Here are all the laboratory tests we offer:

| Test ID | Name | Category | Description | Price (₹) |
|---------|------|----------|-------------|-----------|
| T001 | CBC | Blood | Complete Blood Count test | 500 |
| T002 | Lipid Profile | Blood | Measures cholesterol levels | 800 |
| T003 | Thyroid Profile | Blood | Measures thyroid hormones | 900 |
| T004 | Urine Routine | Urine | Routine urine examination | 300 |
| T005 | Urine Culture | Urine | Checks for bacterial growth in urine | 600 |
| T006 | Chest X‑Ray | Imaging | Chest radiography examination | 700 |
| T007 | Ultrasound | Imaging | Ultrasound examination | 1200 |
| T008 | ECG | Imaging | Electrocardiogram examination | 400 |

How can I assist you today?  
- If you’d like to see available appointment slots for a specific test, let me know the **Test ID**.  
- If you want to view your existing bookings, just say “Show my bookings.”  
- If you’d li

In [29]:
select_active_patient("U001")

agent_response = lab_booking_agent.invoke(
    {"messages": [{"role": "user", "content": "Show my bookings."}]},
    context={"user_id": ACTIVE_PATIENT_ID},
    config={"configurable": {"thread_id": ACTIVE_PATIENT_ID}}
)

print("\nU001 booking information:")
print(agent_response["messages"][-1].content)

print("\nU001 trying to cancel U999's booking B001:")
print(process_booking_cancellation("B001", ACTIVE_PATIENT_ID))

select_active_patient("U999")

agent_response = lab_booking_agent.invoke(
    {"messages": [{"role": "user", "content": "Show my bookings."}]},
    context={"user_id": ACTIVE_PATIENT_ID},
    config={"configurable": {"thread_id": ACTIVE_PATIENT_ID}}
)

print("\nU999 booking information:")
print(agent_response["messages"][-1].content)

select_active_patient("U001")


Current Patient: U001

U001 booking information:
You currently have no confirmed bookings.  
Let me know if you’d like to schedule a new test or if there’s anything else I can help with!

U001 trying to cancel U999's booking B001:
Booking not found for your account.
Current Patient: U999

U999 booking information:
Here are your current bookings:

| Booking ID           | Test ID | Slot ID | Status    |
|----------------------|---------|---------|-----------|
| B001                 | T008    | S0081   | Confirmed |
| B002                 | T008    | S0082   | Confirmed |
| B003                 | T008    | S0083   | Confirmed |
| B20261119223542983040 | T002    | S0021   | Confirmed |

Let me know if you’d like to view details, cancel a booking, or book a new test.
Current Patient: U001


In [30]:
select_active_patient("U001")

lab_database_cursor.execute("""
    SELECT test_id, slot_id
    FROM slots
    WHERE status = 'Available'
    LIMIT 1
""")

slot = lab_database_cursor.fetchone()

if slot != None:

    lab_database_cursor.execute("SELECT COUNT(*) FROM bookings")
    before = lab_database_cursor.fetchone()[0]

    DEMO_APPROVAL_DECISION = "no"

    result = process_test_booking(slot[0], slot[1], ACTIVE_PATIENT_ID)

    lab_database_cursor.execute("SELECT COUNT(*) FROM bookings")
    after = lab_database_cursor.fetchone()[0]

    print("\nResult:", result)
    print("Bookings Before:", before)
    print("Bookings After:", after)

    if before == after:
        print("PASS: Rejection made no database changes.")
    else:
        print("FAIL: Database changed.")

else:
    print("No available slot found.")

DEMO_APPROVAL_DECISION = None


Current Patient: U001

Action: Book Test
Details: CBC | Rs. 500.0 | 2026-11-20 09:00
Automatic Approval: no
Approval: Rejected

Result: Booking rejected. No changes made.
Bookings Before: 8
Bookings After: 8
PASS: Rejection made no database changes.


In [31]:
select_active_patient("U001")

lab_database_cursor.execute("""
    SELECT test_id, slot_id
    FROM slots
    WHERE status = 'Available'
    LIMIT 1
""")

slot = lab_database_cursor.fetchone()

if slot != None:

    DEMO_APPROVAL_DECISION = "yes"

    result = process_test_booking(slot[0], slot[1], ACTIVE_PATIENT_ID)

    print("\nBooking Result:", result)

    lab_database_cursor.execute("""
        SELECT booking_id
        FROM bookings
        WHERE user_id = ? AND test_id = ?
        AND status = 'Confirmed'
        ORDER BY rowid DESC
        LIMIT 1
    """, (ACTIVE_PATIENT_ID, slot[0]))

    booking_record = lab_database_cursor.fetchone()

    if booking_record != None:

        result = process_booking_cancellation(booking_record[0], ACTIVE_PATIENT_ID)

        print("\nCancellation Result:", result)

else:
    print("No available slot found.")

DEMO_APPROVAL_DECISION = None


Current Patient: U001

Action: Book Test
Details: CBC | Rs. 500.0 | 2026-11-20 09:00
Automatic Approval: yes
Approval: Granted

----- BOOKING RECEIPT -----
Booking ID: B20261121221319328300
Patient ID: U001
Test ID: T001
Slot ID: S0011
Status: Confirmed

Booking Result: Booking successful!

Action: Cancel Booking
Details: B20261121221319328300
Automatic Approval: yes
Approval: Granted

Cancellation Result: Booking cancelled successfully!


In [32]:
user_message = "My email is rajnandini@example.com"

redacted_message = redact_email(user_message)

print("Original:", user_message)
print("Sent to AI:", redacted_message)


SIMULATE_ONE_SLOT_CHECK_FAILURE = True
EXECUTED_TOOL_HISTORY = []

select_active_patient("U001")

agent_response = lab_booking_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Check available slots for test T001. If the tool fails temporarily, retry the check."
            }
        ]
    },
    context={"user_id": ACTIVE_PATIENT_ID},
    config={
        "configurable": {
            "thread_id": ACTIVE_PATIENT_ID
        }
    }
)

print("\nTool Calls:", EXECUTED_TOOL_HISTORY)
print("\nAgent Response:")
print(agent_response["messages"][-1].content)


Original: My email is rajnandini@example.com
Sent to AI: My email is [REDACTED]
Current Patient: U001

Tool Calls: ['check_slots', 'check_slots']

Agent Response:
Here are the available slots for the **Complete Blood Count (CBC)** test (T001):

| Slot ID | Date | Time |
|---------|------|------|
| S0011 | 2026‑11‑20 | 09:00 |
| S0012 | 2026‑11‑20 | 10:00 |
| S0013 | 2026‑11‑20 | 11:00 |

Would you like to book one of these slots? If so, let me know the Slot ID you prefer, and I’ll need your approval before proceeding.


In [33]:
lab_evaluation_results = []

def record_evaluation_result(scenario, expected_tools, actual_tools,
               expected_change, actual_change, passed):

    lab_evaluation_results.append([
        scenario,
        expected_tools,
        actual_tools,
        expected_change,
        actual_change,
        "PASS" if passed else "FAIL"
    ])


lab_database_cursor.execute("""
    SELECT test_id, slot_id
    FROM slots
    WHERE status = 'Available'
    GROUP BY test_id
""")

available_slot_rows = lab_database_cursor.fetchall()


if len(available_slot_rows) >= 3:

    EXECUTED_TOOL_HISTORY.clear()

    before = lab_database_cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    select_active_patient("U001")

    check_slots.invoke({"test_id": available_slot_rows[0][0]})

    after = lab_database_cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    record_evaluation_result(
        "Availability",
        "check_slots",
        str(EXECUTED_TOOL_HISTORY),
        "No change",
        "No change" if before == after else "Changed",
        "check_slots" in EXECUTED_TOOL_HISTORY and before == after
    )


    EXECUTED_TOOL_HISTORY.clear()

    DEMO_APPROVAL_DECISION = "yes"

    result = process_test_booking(
        available_slot_rows[0][0],
        available_slot_rows[0][1],
        ACTIVE_PATIENT_ID
    )

    booking1 = lab_database_cursor.execute("""
        SELECT booking_id
        FROM bookings
        WHERE user_id = 'U001'
        AND test_id = ?
        AND status = 'Confirmed'
    """, (available_slot_rows[0][0],)).fetchone()

    record_evaluation_result(
        "Booking 1",
        "book_test",
        str(EXECUTED_TOOL_HISTORY),
        "Booking added",
        "Booking added" if booking1 else "No booking",
        booking1 != None
    )


    EXECUTED_TOOL_HISTORY.clear()

    select_active_patient("U999")

    result = process_test_booking(
        available_slot_rows[1][0],
        available_slot_rows[1][1],
        ACTIVE_PATIENT_ID
    )

    booking2 = lab_database_cursor.execute("""
        SELECT booking_id
        FROM bookings
        WHERE user_id = 'U999'
        AND test_id = ?
        AND status = 'Confirmed'
    """, (available_slot_rows[1][0],)).fetchone()

    record_evaluation_result(
        "Booking 2",
        "book_test",
        str(EXECUTED_TOOL_HISTORY),
        "Booking added",
        "Booking added" if booking2 else "No booking",
        booking2 != None
    )


    EXECUTED_TOOL_HISTORY.clear()

    select_active_patient("U001")

    before = lab_database_cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    DEMO_APPROVAL_DECISION = "no"

    result = process_test_booking(
        available_slot_rows[2][0],
        available_slot_rows[2][1],
        ACTIVE_PATIENT_ID
    )

    after = lab_database_cursor.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    record_evaluation_result(
        "Booking Rejection",
        "book_test",
        str(EXECUTED_TOOL_HISTORY),
        "No change",
        "No change" if before == after else "Changed",
        before == after and "rejected" in result.lower()
    )


    EXECUTED_TOOL_HISTORY.clear()

    result = process_test_booking(
        available_slot_rows[0][0],
        available_slot_rows[0][1],
        ACTIVE_PATIENT_ID
    )

    record_evaluation_result(
        "Slot Conflict",
        "book_test",
        str(EXECUTED_TOOL_HISTORY),
        "No change",
        "No change",
        "not available" in result.lower()
    )


    EXECUTED_TOOL_HISTORY.clear()

    DEMO_APPROVAL_DECISION = "yes"

    select_active_patient("U001")

    if booking1 != None:

        result = process_booking_cancellation(booking1[0], ACTIVE_PATIENT_ID)

        cancelled = lab_database_cursor.execute("""
            SELECT status FROM bookings
            WHERE booking_id = ?
        """, (booking1[0],)).fetchone()

        record_evaluation_result(
            "Cancellation",
            "cancel_booking",
            str(EXECUTED_TOOL_HISTORY),
            "Booking cancelled",
            "Booking cancelled" if cancelled[0] == "Cancelled" else "Not cancelled",
            cancelled[0] == "Cancelled"
        )


    EXECUTED_TOOL_HISTORY.clear()

    search_result = general_info_search.invoke({
        "query": "Why do some blood tests require fasting?"
    })

    EXECUTED_TOOL_HISTORY.append("web_search")

    record_evaluation_result(
        "General Information",
        "web_search",
        str(EXECUTED_TOOL_HISTORY),
        "No change",
        "No change",
        "web_search" in EXECUTED_TOOL_HISTORY
    )


    EXECUTED_TOOL_HISTORY.clear()

    agent_response = lab_booking_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "I have a fever. Which blood test should I take?"
                }
            ]
        },
        context={"user_id": "U001"},
        config={
            "configurable": {
                "thread_id": "U001"
            }
        }
    )

    write_tools = ["book_test", "cancel_booking"]

    used_write_tool = False

    for tool_name in EXECUTED_TOOL_HISTORY:
        if tool_name in write_tools:
            used_write_tool = True

    record_evaluation_result(
        "Symptom Question",
        "No booking/cancellation",
        str(EXECUTED_TOOL_HISTORY),
        "No change",
        "No change",
        used_write_tool == False
    )


else:
    print("Not enough available test slots to run all scenarios.")


DEMO_APPROVAL_DECISION = None


print("\nLAB TEST BOOKING ASSISTANT - EVALUATION")

print("-" * 120)

print(
    "{:<22} {:<25} {:<25} {:<20} {:<15} {:<10}".format(
        "Scenario",
        "Expected Tools",
        "Actual Tools",
        "Expected DB Change",
        "Actual DB Change",
        "Result"
    )
)

print("-" * 120)

for row in lab_evaluation_results:

    print(
        "{:<22} {:<25} {:<25} {:<20} {:<15} {:<10}".format(
            str(row[0])[:21],
            str(row[1])[:24],
            str(row[2])[:24],
            str(row[3])[:19],
            str(row[4])[:14],
            row[5]
        )
    )

print("-" * 120)

Current Patient: U001

Action: Book Test
Details: CBC | Rs. 500.0 | 2026-11-20 09:00
Automatic Approval: yes
Approval: Granted

----- BOOKING RECEIPT -----
Booking ID: B20261121221330436556
Patient ID: U001
Test ID: T001
Slot ID: S0011
Status: Confirmed
Current Patient: U999
Current Patient: U001

Action: Book Test
Details: Thyroid Profile | Rs. 900.0 | 2026-11-20 09:00
Automatic Approval: no
Approval: Rejected
Current Patient: U001

Action: Cancel Booking
Details: B20261121221330436556
Automatic Approval: yes
Approval: Granted

LAB TEST BOOKING ASSISTANT - EVALUATION
------------------------------------------------------------------------------------------------------------------------
Scenario               Expected Tools            Actual Tools              Expected DB Change   Actual DB Change Result    
------------------------------------------------------------------------------------------------------------------------
Availability           check_slots               ['check_sl

In [34]:
print("TESTS TABLE")

lab_database_cursor.execute("SELECT * FROM tests")

for row in lab_database_cursor.fetchall():
    print(row)


print("\nSLOTS TABLE")

lab_database_cursor.execute("SELECT * FROM slots")

for row in lab_database_cursor.fetchall():
    print(row)


print("\nBOOKINGS TABLE")

lab_database_cursor.execute("SELECT * FROM bookings")

for row in lab_database_cursor.fetchall():
    print(row)

TESTS TABLE
('T001', 'CBC', 'Blood', 'Complete Blood Count test', 500.0)
('T002', 'Lipid Profile', 'Blood', 'Measures cholesterol levels', 800.0)
('T003', 'Thyroid Profile', 'Blood', 'Measures thyroid hormones', 900.0)
('T004', 'Urine Routine', 'Urine', 'Routine urine examination', 300.0)
('T005', 'Urine Culture', 'Urine', 'Checks for bacterial growth in urine', 600.0)
('T006', 'Chest X-Ray', 'Imaging', 'Chest radiography examination', 700.0)
('T007', 'Ultrasound', 'Imaging', 'Ultrasound examination', 1200.0)
('T008', 'ECG', 'Imaging', 'Electrocardiogram examination', 400.0)

SLOTS TABLE
('S0011', 'T001', '2026-11-20', '09:00', 'Available')
('S0012', 'T001', '2026-11-20', '10:00', 'Available')
('S0013', 'T001', '2026-11-20', '11:00', 'Available')
('S0021', 'T002', '2026-11-20', '09:00', 'Booked')
('S0022', 'T002', '2026-11-20', '10:00', 'Available')
('S0023', 'T002', '2026-11-20', '11:00', 'Available')
('S0031', 'T003', '2026-11-20', '09:00', 'Available')
('S0032', 'T003', '2026-11-20'

In [35]:
lab_database_connection.close()
print("Database connection closed successfully!")

Database connection closed successfully!
